In [1]:
!pip install torch torchtext transformers sentencepiece pandas tqdm datasets

In [2]:
from datasets import load_dataset, DatasetDict, Dataset
import pandas as pd
import ast
import gc
import datasets
from tqdm import tqdm
import time
from transformers import GPT2TokenizerFast, GPT2LMHeadModel, AutoTokenizerFast
import torch
import torch.nn as nn
import torch.optim as optim
import os
from torch.utils.data import Dataset, DataLoader, random_split

W1022 13:02:42.627000 81666 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1022 13:02:42.646000 81666 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [3]:
#Loading the dataset
dataset = load_dataset("prognosis/symptoms_disease_v1")
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'reference', 'output', 'instruction'],
        num_rows: 10110
    })
})

In [4]:
#converting t0 pandas df
updated_data = [{'Input' :item['instruction'], 'Disease': item['output']} for item in dataset['train']]
df = pd.DataFrame(updated_data)
df.head()

,Input,Disease
0,What are the symptoms of hypertensive disease?,The following are the symptoms of hypertensive...
1,I am having the following symptoms: pain ches...,The symptoms listed indicates that the patient...
2,What are the symptoms of diabetes?,The following are the symptoms of diabetes: po...
3,"I am having the following symptoms: polyuria, ...",The symptoms listed indicates that the patient...
4,What are the symptoms of depressive disorder?,The following are the symptoms of depressive d...


In [5]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
device

device(type='cuda')

In [6]:
#The tokenizer turns text to numbers
tokenizer = GPT2TokenizerFast.from_pretrained('distilgpt2')

In [7]:
# The transformers
model = GPT2LMHeadModel.from_pretrained('distilgpt2').to(device)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [8]:
from torch.utils.data import Dataset

# Dataset Prep
from torch.utils.data import Dataset

class LanguageDataset(Dataset):
    """
    An extension of the Dataset object to:
      - Make training loop cleaner
      - Make ingestion easier from pandas df's
    """
    def __init__(self, df, tokenizer):
        self.labels = df.columns
        self.data = df.to_dict(orient='records')
        self.tokenizer = tokenizer
        x = self.fittest_max_length(df)  
        self.max_length = x

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx][self.labels[0]]
        y = self.data[idx][self.labels[1]]
        text = f"{x} | {y}"
        
        # CHANGED LINE: Calling self.tokenizer directly works perfectly 
        # on the basic GPT2Tokenizer class!
        tokens = self.tokenizer(
            text, 
            return_tensors='pt', 
            max_length=self.max_length, 
            padding='max_length', 
            truncation=True
        )
        return tokens

    def fittest_max_length(self, df):  
        """
        Smallest power of two larger than the longest term in the data set.
        Important to set up max length to speed training time.
        """
        max_length = max(len(max(df[self.labels[0]], key=len)), len(max(df[self.labels[1]], key=len)))
        x = 2
        while x < max_length: 
            x = x * 2
        return x


# CRITICAL FIX: This must be completely unindented (moved to the far left)
# so it runs outside of the class block.
data_sample = LanguageDataset(df, tokenizer)
data_sample.max_length = 32


In [12]:
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)

In [11]:
# Splitting dataset into training and validation sets
train_size = int(0.8 * len(data_sample))
valid_size = len(data_sample) - train_size
train_data, valid_data = random_split(data_sample, [train_size, valid_size])

In [10]:
# Set the number of epochs
num_epochs = 2
# Model params
BATCH_SIZE = 4
# Training parameters
batch_size = BATCH_SIZE
model_name = 'distilgpt2'
gpu = 0
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = optim.Adam(model.parameters(), lr=5e-4)
tokenizer.pad_token = tokenizer.eos_token
# Init a results dataframe
results = pd.DataFrame(columns=['epoch', 'transformer', 'batch_size', 'gpu',
                                'training_loss', 'validation_loss', 'epoch_duration_sec'])

In [13]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [14]:
# Prevent memory fragmentation inside the caching allocator
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Initialize the GradScaler for Mixed Precision training
scaler = torch.cuda.amp.GradScaler()

# Set how many batches to calculate before tweaking the model weights
# (e.g., 4 means you accumulate gradients across 4 small steps)
accumulation_steps = 4 

# Clear out any leftover junk in your GPU VRAM before starting
gc.collect()
torch.cuda.empty_cache()

# The training loop
for epoch in range(num_epochs):
    start_time = time.time()  # Start the timer for the epoch
    
    # --- TRAINING PHASE ---
    model.train()
    epoch_training_loss = 0
    train_iterator = tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{num_epochs} Batch Size: {batch_size}, Transformer: {model_name}")
    
    optimizer.zero_grad() # Clear gradients once at the start of the epoch
    
    for i, batch in enumerate(train_iterator):
        inputs = batch['input_ids'].squeeze(1).to(device)
        targets = inputs.clone()
        
        # 1. Forward Pass using 16-bit Mixed Precision
        with torch.cuda.amp.autocast():
            outputs = model(input_ids=inputs, labels=targets)
            # Scale the loss downward according to your accumulation steps
            loss = outputs.loss / accumulation_steps
            
        # 2. Backward Pass using scaled gradients to fit in 6GB VRAM
        scaler.scale(loss).backward()
        
        # 3. Update weights only after reaching your accumulation target step
        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        # Display the real loss value on your progress bar
        real_loss_value = loss.item() * accumulation_steps
        train_iterator.set_postfix({'Training Loss': real_loss_value})
        epoch_training_loss += real_loss_value
        
    avg_epoch_training_loss = epoch_training_loss / len(train_iterator)
    
    # --- VALIDATION PHASE ---
    model.eval()
    epoch_validation_loss = 0
    total_loss = 0
    valid_iterator = tqdm(valid_loader, desc=f"Validation Epoch {epoch+1}/{num_epochs}")
    
    with torch.no_grad():
        for batch in valid_iterator:
            inputs = batch['input_ids'].squeeze(1).to(device)
            targets = inputs.clone()
            
            # Use autocast during validation too to save memory
            with torch.cuda.amp.autocast():
                outputs = model(input_ids=inputs, labels=targets)
                loss = outputs.loss
                
            total_loss += loss.item()  # Convert tensor to scalar
            valid_iterator.set_postfix({'Validation Loss': loss.item()})
            epoch_validation_loss += loss.item()
            
    avg_epoch_validation_loss = epoch_validation_loss / len(valid_loader)
    end_time = time.time()  # End the timer for the epoch
    epoch_duration_sec = end_time - start_time  # Calculate the duration in seconds
    
    new_row = {'transformer': model_name,
               'batch_size': batch_size,
               'gpu': gpu,
               'epoch': epoch+1,
               'training_loss': avg_epoch_training_loss,
               'validation_loss': avg_epoch_validation_loss,
               'epoch_duration_sec': epoch_duration_sec}  # Add epoch_duration to the dataframe
    results.loc[len(results)] = new_row
    print(f"Epoch: {epoch+1}, Validation Loss: {total_loss/len(valid_loader)}")

/tmp/ipykernel_81666/2390091052.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Training Epoch 1/2 Batch Size: 4, Transformer: distilgpt2:   0%|          | 0/2022 [00:00<?, ?it/s]/tmp/ipykernel_81666/2390091052.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Training Epoch 1/2 Batch Size: 4, Transformer: distilgpt2: 100%|██████████| 2022/2022 [01:10<00:00, 28.57it/s, Training Loss=0.247]
Validation Epoch 1/2:   0%|          | 0/506 [00:00<?, ?it/s]/tmp/ipykernel_81666/2390091052.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Valida

Epoch: 1, Validation Loss: 0.24443528533217465


Training Epoch 2/2 Batch Size: 4, Transformer: distilgpt2:   0%|          | 0/2022 [00:00<?, ?it/s]/tmp/ipykernel_81666/2390091052.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Training Epoch 2/2 Batch Size: 4, Transformer: distilgpt2: 100%|██████████| 2022/2022 [01:10<00:00, 28.76it/s, Training Loss=0.191]
Validation Epoch 2/2:   0%|          | 0/506 [00:00<?, ?it/s]/tmp/ipykernel_81666/2390091052.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Validation Epoch 2/2: 100%|██████████| 506/506 [00:04<00:00, 117.68it/s, Validation Loss=0.31] 

Epoch: 2, Validation Loss: 0.23515937972092346


In [ ]:
gc.collect()
torch.cuda.empty_cache()